In [ ]:
#| default_exp icon1

# icon1
> all the functionalities required to manipulate markdown files i.e edit , open when clicked fromt he d3.js note gallery view

In [ ]:
#| export
from dataclasses import dataclass
from typing import Any, Callable

from fasthtml.common import *
from monsterui.all import *
import re

## Visualization layer

This Python code implements a dynamic, Jupyter-like notebook interface using FastHTML and HTMX. It parses Markdown into distinct text and code "cells," styling them with MonsterUI. Users can toggle between a live-updating editor and a rendered view, enabling seamless, interactive document creation without full page reloads.

In [ ]:
#| export
NOTEBOOK_INITIAL_MD = """# Notebook
This is markdown with **MonsterUI** styling.

```python
def hello():
    print("Edit below or click Edit for source.")
```

Click **Edit** for the raw markdown; **Done** returns here."""


def NotebookCell(content, cell_type="markdown"):
    if cell_type == "code":
        return Div(
            Div(P("In [ ]:", cls="text-xs text-muted-foreground absolute -left-12 top-4 font-mono")),
            Card(render_md(content), cls="bg-muted/30 border-l-4 border-l-primary relative"),
            cls="mb-6 ml-12",
        )
    return Div(
        render_md(content),
        cls="mb-6 px-4 prose prose-slate max-w-none",
    )


def parse_to_notebook(md_text):
    pattern = r'(```[\s\S]*?```)'
    parts = re.split(pattern, md_text)

    cells = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if part.startswith("```"):
            cells.append(NotebookCell(part, cell_type="code"))
        else:
            cells.append(NotebookCell(part, cell_type="markdown"))
    return Div(*cells, id="notebook-view")


def notebook_editor_panel(md: str):
    return Div(
        H3("Markdown", cls="mb-4"),
        P("Preview updates as you type. **Done** shows the notebook view.", cls="text-sm opacity-70 mb-3"),
        Textarea(
            md,
            id="md-input",
            name="content",
            cls="textarea textarea-bordered w-full min-h-[10rem] flex-1 max-h-full font-mono text-sm resize-y",
            hx_post="/icon/render",
            hx_trigger="keyup changed delay:300ms",
            hx_swap="none",
        ),
        cls="space-y-2 min-w-0 w-full py-2 flex-1 min-h-0 flex flex-col",
    )


def icon_panel(md: str, editor_open: bool):
    """Root fragment; id=icon-panel is the HTMX swap target."""
    edit_btn = Button(
        "Edit",
        cls="btn btn-outline btn-primary btn-sm shrink-0",
        hx_get="/icon/panel?edit=1",
        hx_target="#icon-panel",
        hx_swap="outerHTML",
    )
    done_btn = Button(
        "Done",
        cls="btn btn-outline btn-primary btn-sm shrink-0",
        hx_get="/icon/panel?edit=0",
        hx_target="#icon-panel",
        hx_swap="outerHTML",
    )
    header = Div(
        H2("Notebook", cls="text-2xl font-bold truncate min-w-0"),
        Div(
            done_btn if editor_open else edit_btn,
            cls="flex items-center gap-2 shrink-0",
        ),
        cls="flex flex-row justify-between items-center gap-2 mb-2 w-full",
    )
    if editor_open:
        body = notebook_editor_panel(md)
    else:
        body = Div(
            parse_to_notebook(md),
            cls="w-full min-w-0 py-2 flex-1 min-h-0 overflow-y-auto",
        )
    return Div(
        header,
        body,
        id="icon-panel",
        cls="relative p-4 w-full h-full min-h-0 flex flex-col overflow-hidden",
    )


In [ ]:
#| export
@dataclass(frozen=True)
class Icon1VizLayer:
    """Injectable notebook UI (same role as ``visualizations.icon1.viz`` in string therapy1)."""

    notebook_initial_md: str
    icon_panel: Callable[[str, bool], Any]
    parse_to_notebook: Callable[[str], Any]


def default_icon1_viz_layer() -> Icon1VizLayer:
    return Icon1VizLayer(
        notebook_initial_md=NOTEBOOK_INITIAL_MD,
        icon_panel=icon_panel,
        parse_to_notebook=parse_to_notebook,
    )

## Controller Layer

This code acts as the controller for the notebook visualization layer. It manages the "source of truth" for the markdown content and handles the state(weather the user is currently editing or viewing) HEre is the breakdown of how it works 

### 1. State management (the session)
the code uses a `session` dictionary to keep the notebook's data alive. This means the content is per-user and temporary. If they clear their browser cookies or the session expires, the notebook resets. 
> `SESSION_MD_KEY` : Stores the actual markdown text the user types

> `SESSION_EDITOR_KEY`: A boolean (True/False) that remembers if the user was in Edit mode or view mode

### 2. Helper Logic: _md
``` python
def _md(session: dict) -> str:
    return session.get(SESSION_MD_KEY) or NOTEBOOK_INITIAL_MD
```
This is a safety check. It tries to grab the user's saved markdown. If the session is empty (like when they first land on the page), it falls back to the `NOTEBOOK_INITIAL_MD` default text### 2. Helper Logic: _md
``` python
def _md(session: dict) -> str:
    return session.get(SESSION_MD_KEY) or NOTEBOOK_INITIAL_MD
```
This is a safety check. It tries to grab the user's saved markdown. If the session is empty (like when they first land on the page), it falls back to the `NOTEBOOK_INITIAL_MD` default text

### 3. The Routes(the "wiring")
`GET /icon/panel`
This handles the toggle button logic. 
* When you click "Edit," it hits this route with edit=1. The session updates `SESSION_EDITOR_KEY` to `True`
* When you click "Done", it hits this route with `edit=0`
* it then returns the `icon_panel` HTML, which decides wheather to show the `Textarea` (editor) or the `parse_to_notebook` (rendered cells)

`POST icon/render`
This is the Live update engine
* Remember the `hx_trigger="keyup changed delay:300ms` from the vizualization layer. Every time you pause typing for 300ms, it sends the text here
* it saves that text into the session (`session[SESSION_MD_KEY] = content`).
* It then returns the Notebook view (the rendered cells ) so the UI can update immediately.

In [ ]:
#| export
SESSION_MD_KEY = "notebook_md"
SESSION_EDITOR_KEY = "notebook_editor_open"
NOTEBOOK_ICON_ID = 1


def _md(session: dict, layer: Icon1VizLayer) -> str:
    return session.get(SESSION_MD_KEY) or layer.notebook_initial_md


def viz(sid: str, session: dict, viz_layer: Icon1VizLayer | None = None):
    layer = viz_layer or default_icon1_viz_layer()
    session[SESSION_EDITOR_KEY] = False
    return layer.icon_panel(_md(session, layer), False)


def register_routes(app, viz_layer: Icon1VizLayer | None = None):
    layer = viz_layer or default_icon1_viz_layer()

    @app.get("/icon/panel")
    def notebook_panel_route(session, edit: str | None = None):
        if edit == "1":
            session[SESSION_EDITOR_KEY] = True
        elif edit == "0":
            session[SESSION_EDITOR_KEY] = False
        return layer.icon_panel(_md(session, layer), session.get(SESSION_EDITOR_KEY, False))

    @app.post("/icon/render")
    def notebook_render(content: str, session):
        session[SESSION_MD_KEY] = content
        return layer.parse_to_notebook(content)